# **Quaternion-Aware CoHAtNet**

 **1. INSTALLATION**

In [ ]:
!pip install transforms3d einops kornia

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 101.4 MB/s eta 0:00:00


**2. MOUNT DRIVE**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


**3. IMPORTS**

In [ ]:
import os
import random
import json
import pickle
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from einops import rearrange
from einops.layers.torch import Rearrange
from sklearn.model_selection import GroupKFold
from tqdm import tqdm
import transforms3d.quaternions as txq
from kornia.geometry.conversions import quaternion_to_rotation_matrix

**4. REPRODUCIBILITY**

In [ ]:
def set_seed(seed=42):
    """Set random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(42)

**5. GLOBAL HOMOGRAPHY LOSS**

In [ ]:
class GlobalHomographyLoss(nn.Module):
    """
    Geometric consistency loss.
    IMPORTANT: translations must be in the original metric scale.
    """
    def __init__(self, xmin, xmax, device='cpu'):
        super().__init__()

        xmin = max(float(xmin), 1e-6)
        xmax = max(float(xmax), xmin + 1e-6)

        xmin = torch.tensor(xmin, dtype=torch.float32, device=device)
        xmax = torch.tensor(xmax, dtype=torch.float32, device=device)

        self.B_weight = torch.log(xmin / xmax) / (xmax - xmin)
        self.C_weight = xmin * xmax
        self.c_n = torch.tensor([0, 0, -1], dtype=torch.float32, device=device).view(3, 1)
        self.eye = torch.eye(3, device=device)

    def forward(self, batch):
        A, B, C = compute_ABC(
            batch['w_t_c'],
            batch['c_R_w'],
            batch['w_t_chat'],
            batch['chat_R_w'],
            self.c_n,
            self.eye
        )

        error = A + B * self.B_weight + C / self.C_weight
        error = error.diagonal(dim1=1, dim2=2).sum(dim=1).mean()
        return error


def compute_ABC(w_t_c, c_R_w, w_t_chat, chat_R_w, c_n, eye):
    """
    Compute A, B, and C matrices given estimated and ground-truth poses.
    """
    w_t_c = w_t_c.float()
    c_R_w = c_R_w.float()
    w_t_chat = w_t_chat.float()
    chat_R_w = chat_R_w.float()
    c_n = c_n.float()
    eye = eye.float()

    chat_t_c = chat_R_w @ (w_t_c - w_t_chat)
    chat_R_c = chat_R_w @ c_R_w.transpose(1, 2)

    A = eye - chat_R_c
    C = c_n @ chat_t_c.transpose(1, 2)
    B = C @ A
    A = A @ A.transpose(1, 2)
    B = B + B.transpose(1, 2)
    C = C @ C.transpose(1, 2)

    return A, B, C


**6. POSE / DEPTH UTILITIES**

In [ ]:
def read_pose_file(pose_path):
    """Read a 3x4 pose matrix and return it flattened to 12 values."""
    pose = np.loadtxt(pose_path).flatten()[:12].astype(np.float32)
    return pose


def pose_matrix_to_pose_vector(poses_in, mean_t, std_t, align_R=None, align_t=None, align_s=1.0):
    """
    Convert pose matrices into [tx, ty, tz, qw, qx, qy, qz].
    Translation is normalized using mean_t and std_t.
    """
    if align_R is None:
        align_R = np.eye(3, dtype=np.float32)
    if align_t is None:
        align_t = np.zeros(3, dtype=np.float32)

    poses_in = np.asarray(poses_in, dtype=np.float32)
    if poses_in.ndim == 1:
        poses_in = poses_in[None, :]

    mean_t = np.asarray(mean_t, dtype=np.float32)
    std_t = np.asarray(std_t, dtype=np.float32)
    std_t = np.where(std_t < 1e-8, 1.0, std_t)

    poses_out = np.zeros((len(poses_in), 7), dtype=np.float32)
    poses_out[:, 0:3] = poses_in[:, [3, 7, 11]]

    for i in range(len(poses_out)):
        R = poses_in[i].reshape((3, 4))[:3, :3]
        q = txq.mat2quat(np.dot(align_R, R))
        q *= np.sign(q[0]) if q[0] != 0 else 1.0
        poses_out[i, 3:] = q

        t = poses_out[i, :3] - align_t
        poses_out[i, :3] = align_s * np.dot(align_R, t[:, np.newaxis]).squeeze()

    poses_out[:, :3] -= mean_t
    poses_out[:, :3] /= std_t
    return poses_out


def denormalize_translation(t_norm, mean_t, std_t):
    """Convert normalized translation back to metric scale."""
    mean_t = torch.as_tensor(mean_t, dtype=t_norm.dtype, device=t_norm.device)
    std_t = torch.as_tensor(std_t, dtype=t_norm.dtype, device=t_norm.device)
    return t_norm * std_t + mean_t


def compute_train_translation_stats(base_dataset, train_indices):
    """
    Compute translation mean/std from TRAIN fold only.
    """
    translations = []

    for idx in train_indices:
        pose_path = base_dataset.samples[idx]['pose_path']
        pose = read_pose_file(pose_path)
        translations.append(pose[[3, 7, 11]])

    translations = np.asarray(translations, dtype=np.float32)
    mean_t = np.mean(translations, axis=0)
    std_t = np.std(translations, axis=0)
    std_t = np.where(std_t < 1e-8, 1.0, std_t)

    return mean_t, std_t


def compute_train_depth_stats(base_dataset, train_indices, max_samples=100):
    """
    Compute xmin/xmax from TRAIN fold only.
    """
    if len(train_indices) == 0:
        return 1.0, 2.0

    sampled_indices = list(train_indices)
    if len(sampled_indices) > max_samples:
        sampled_indices = random.sample(sampled_indices, max_samples)

    all_depths = []

    for idx in tqdm(sampled_indices, desc='Computing train depth stats'):
        depth_path = base_dataset.samples[idx]['depth_path']
        depth_image = cv2.imread(depth_path, cv2.IMREAD_UNCHANGED)

        if depth_image is None:
            continue

        valid = depth_image[depth_image > 0]
        if valid.size > 0:
            all_depths.extend(valid.flatten())

    if len(all_depths) == 0:
        return 1.0, 2.0

    all_depths = np.asarray(all_depths, dtype=np.float32)
    xmin = np.percentile(all_depths, 2.5)
    xmax = np.percentile(all_depths, 97.5)

    xmin = max(float(xmin), 1e-6)
    xmax = max(float(xmax), xmin + 1e-6)

    return xmin, xmax



**7. DATASET**

In [ ]:
class BaseFireDataset(Dataset):
    """
    Base dataset that stores file paths and sequence names only.
    No fold-specific normalization is applied here.
    """
    def __init__(self, root_dir):
        self.root_dir = root_dir
        self.seqs = sorted([
            seq for seq in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, seq))
        ])
        self.samples = self._load_samples()
        self.sample_groups = [sample['seq_name'] for sample in self.samples]

    def _load_samples(self):
        samples = []

        for seq_folder in self.seqs:
            seq_path = os.path.join(self.root_dir, seq_folder)

            color_files = sorted([f for f in os.listdir(seq_path) if f.endswith('.color.png')])
            depth_files = sorted([f for f in os.listdir(seq_path) if f.endswith('.depth.png')])
            pose_files = sorted([f for f in os.listdir(seq_path) if f.endswith('.pose.txt')])

            for c, d, p in zip(color_files, depth_files, pose_files):
                samples.append({
                    'seq_name': seq_folder,
                    'color_path': os.path.join(seq_path, c),
                    'depth_path': os.path.join(seq_path, d),
                    'pose_path': os.path.join(seq_path, p)
                })

        return samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


class FoldFireDataset(Dataset):
    """
    Fold-specific dataset.
    Uses TRAIN-fold statistics to normalize poses.
    RGB only as input.
    """
    def __init__(self, base_dataset, indices, mean_t, std_t, transform=None):
        self.base_dataset = base_dataset
        self.indices = list(indices)
        self.mean_t = mean_t
        self.std_t = std_t
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        sample = self.base_dataset.samples[idx]

        color_image = cv2.imread(sample['color_path'], cv2.IMREAD_COLOR)
        if color_image is None:
            raise ValueError(f"Could not read image: {sample['color_path']}")

        color_image = cv2.cvtColor(color_image, cv2.COLOR_BGR2RGB)

        if self.transform:
            color_image = self.transform(color_image)

        raw_pose = read_pose_file(sample['pose_path'])
        pose_vector = pose_matrix_to_pose_vector(
            raw_pose,
            mean_t=self.mean_t,
            std_t=self.std_t
        )[0]

        return color_image, torch.tensor(pose_vector, dtype=torch.float32)



**8. IMAGE TRANSFORM**

In [ ]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

**9. COHATNET BACKBONE BLOCKS**

In [ ]:
def conv_3x3_bn(inp, oup, image_size, downsample=False):
    stride = 1 if downsample is False else 2
    return nn.Sequential(
        nn.Conv2d(inp, oup, 3, stride, 1, bias=False),
        nn.BatchNorm2d(oup),
        nn.GELU()
    )


class PreNorm(nn.Module):
    def __init__(self, dim, fn, norm):
        super().__init__()
        self.norm = norm(dim)
        self.fn = fn

    def forward(self, x, **kwargs):
        return self.fn(self.norm(x), **kwargs)


class SE(nn.Module):
    def __init__(self, inp, oup, expansion=0.25):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(oup, int(inp * expansion), bias=False),
            nn.GELU(),
            nn.Linear(int(inp * expansion), oup, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y


class FeedForward(nn.Module):
    def __init__(self, dim, hidden_dim, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class MBConv(nn.Module):
    def __init__(self, inp, oup, image_size, downsample=False, expansion=4):
        super().__init__()
        self.downsample = downsample
        stride = 1 if self.downsample is False else 2
        hidden_dim = int(inp * expansion)

        if self.downsample:
            self.pool = nn.MaxPool2d(3, 2, 1)
            self.proj = nn.Conv2d(inp, oup, 1, 1, 0, bias=False)

        if expansion == 1:
            self.conv = nn.Sequential(
                nn.Conv2d(hidden_dim, hidden_dim, 3, stride, 1, groups=hidden_dim, bias=False),
                nn.BatchNorm2d(hidden_dim),
                nn.GELU(),
                nn.Conv2d(hidden_dim, oup, 1, 1, 0, bias=False),
                nn.BatchNorm2d(oup),
            )
        else:
            self.conv = nn.Sequential(
                nn.Conv2d(inp, hidden_dim, 1, stride, 0, bias=False),
                nn.BatchNorm2d(hidden_dim),
                nn.GELU(),
                nn.Conv2d(hidden_dim, hidden_dim, 3, 1, 1, groups=hidden_dim, bias=False),
                nn.BatchNorm2d(hidden_dim),
                nn.GELU(),
                SE(inp, hidden_dim),
                nn.Conv2d(hidden_dim, oup, 1, 1, 0, bias=False),
                nn.BatchNorm2d(oup),
            )

        self.conv = PreNorm(inp, self.conv, nn.BatchNorm2d)

    def forward(self, x):
        if self.downsample:
            return self.proj(self.pool(x)) + self.conv(x)
        else:
            return x + self.conv(x)


class Attention(nn.Module):
    def __init__(self, inp, oup, image_size, heads=8, dim_head=32, dropout=0.0):
        super().__init__()
        inner_dim = dim_head * heads
        project_out = not (heads == 1 and dim_head == inp)

        self.ih, self.iw = image_size
        self.heads = heads
        self.scale = dim_head ** -0.5

        self.relative_bias_table = nn.Parameter(
            torch.zeros((2 * self.ih - 1) * (2 * self.iw - 1), heads)
        )

        coords = torch.meshgrid((torch.arange(self.ih), torch.arange(self.iw)), indexing='ij')
        coords = torch.flatten(torch.stack(coords), 1)
        relative_coords = coords[:, :, None] - coords[:, None, :]

        relative_coords[0] += self.ih - 1
        relative_coords[1] += self.iw - 1
        relative_coords[0] *= 2 * self.iw - 1
        relative_coords = rearrange(relative_coords, 'c h w -> h w c')
        relative_index = relative_coords.sum(-1).flatten().unsqueeze(1)
        self.register_buffer("relative_index", relative_index)

        self.attend = nn.Softmax(dim=-1)
        self.to_qkv = nn.Linear(inp, inner_dim * 3, bias=False)

        self.to_out = nn.Sequential(
            nn.Linear(inner_dim, oup),
            nn.Dropout(dropout)
        ) if project_out else nn.Identity()

    def forward(self, x):
        qkv = self.to_qkv(x).chunk(3, dim=-1)
        q, k, v = map(lambda t: rearrange(t, 'b n (h d) -> b h n d', h=self.heads), qkv)

        dots = torch.matmul(q, k.transpose(-1, -2)) * self.scale

        relative_bias = self.relative_bias_table.gather(
            0, self.relative_index.repeat(1, self.heads)
        )
        relative_bias = rearrange(
            relative_bias,
            '(h w) c -> 1 c h w',
            h=self.ih * self.iw,
            w=self.ih * self.iw
        )

        dots = dots + relative_bias
        attn = self.attend(dots)
        out = torch.matmul(attn, v)
        out = rearrange(out, 'b h n d -> b n (h d)')
        out = self.to_out(out)
        return out


class HAttention(nn.Module):
    def __init__(self, inp, oup, image_size, heads=8, dim_head=32, dropout=0.0):
        super().__init__()
        inner_dim = dim_head * heads
        project_out = not (heads == 1 and dim_head == inp)

        self.ih, self.iw = image_size
        self.heads = heads
        self.scale = dim_head ** -0.5

        self.relative_bias_table = nn.Parameter(
            torch.zeros((2 * self.ih - 1) * (2 * self.iw - 1), heads)
        )

        coords = torch.meshgrid((torch.arange(self.ih), torch.arange(self.iw)), indexing='ij')
        coords = torch.flatten(torch.stack(coords), 1)
        relative_coords = coords[:, :, None] - coords[:, None, :]

        relative_coords[0] += self.ih - 1
        relative_coords[1] += self.iw - 1
        relative_coords[0] *= 2 * self.iw - 1
        relative_coords = rearrange(relative_coords, 'c h w -> h w c')
        relative_index = relative_coords.sum(-1).flatten().unsqueeze(1)
        self.register_buffer("relative_index", relative_index)

        self.attend = nn.Softmax(dim=-1)
        self.to_qk = nn.Linear(inp, inner_dim * 2, bias=False)

        self.to_out = nn.Sequential(
            nn.Linear(oup, oup),
            nn.Dropout(dropout)
        ) if project_out else nn.Identity()

    def forward(self, x, mbconv):
        qk = self.to_qk(x).chunk(2, dim=-1)
        q, k = map(lambda t: rearrange(t, 'b n (h d) -> b h n d', h=self.heads), qk)

        dots = torch.matmul(q, k.transpose(-1, -2)) * self.scale

        relative_bias = self.relative_bias_table.gather(
            0, self.relative_index.repeat(1, self.heads)
        )
        relative_bias = rearrange(
            relative_bias,
            '(h w) c -> 1 c h w',
            h=self.ih * self.iw,
            w=self.ih * self.iw
        )
        dots = dots + relative_bias

        attn = self.attend(dots)

        v = rearrange(mbconv, 'b c ih iw -> b (ih iw) c')
        v = rearrange(v, 'b n (h c) -> b h n c', h=self.heads)

        out = torch.matmul(attn, v)
        out = rearrange(out, 'b h n d -> b n (h d)')
        out = self.to_out(out)
        return out


class Transformer(nn.Module):
    def __init__(self, inp, oup, image_size, heads=8, dim_head=32, downsample=False, dropout=0.0):
        super().__init__()
        hidden_dim = int(inp * 4)

        self.ih, self.iw = image_size
        self.downsample = downsample

        if self.downsample:
            self.pool1 = nn.MaxPool2d(3, 2, 1)
            self.pool2 = nn.MaxPool2d(3, 2, 1)
            self.proj = nn.Conv2d(inp, oup, 1, 1, 0, bias=False)

        self.attn = Attention(inp, oup, image_size, heads, dim_head, dropout)
        self.ff = FeedForward(oup, hidden_dim, dropout)

        self.attn = nn.Sequential(
            Rearrange('b c ih iw -> b (ih iw) c'),
            PreNorm(inp, self.attn, nn.LayerNorm),
            Rearrange('b (ih iw) c -> b c ih iw', ih=self.ih, iw=self.iw)
        )

        self.ff = nn.Sequential(
            Rearrange('b c ih iw -> b (ih iw) c'),
            PreNorm(oup, self.ff, nn.LayerNorm),
            Rearrange('b (ih iw) c -> b c ih iw', ih=self.ih, iw=self.iw)
        )

    def forward(self, x):
        if self.downsample:
            x = self.proj(self.pool1(x)) + self.attn(self.pool2(x))
        else:
            x = x + self.attn(x)
        x = x + self.ff(x)
        return x


class HTransformer(nn.Module):
    def __init__(self, inp, oup, image_size, heads=8, dim_head=32, downsample=False, dropout=0.0):
        super().__init__()
        hidden_dim = int(inp * 4)

        self.ih, self.iw = image_size
        self.downsample = downsample
        self.layer_norm = nn.LayerNorm(inp)
        self.MBConv = MBConv(inp, oup, image_size, downsample)

        if self.downsample:
            self.pool1 = nn.MaxPool2d(3, 2, 1)
            self.pool2 = nn.MaxPool2d(3, 2, 1)
            self.proj = nn.Conv2d(inp, oup, 1, 1, 0, bias=False)

        self.attn = HAttention(inp, oup, image_size, heads, dim_head, dropout)
        self.ff = FeedForward(oup, hidden_dim, dropout)

        self.ff = nn.Sequential(
            Rearrange('b c ih iw -> b (ih iw) c'),
            PreNorm(oup, self.ff, nn.LayerNorm),
            Rearrange('b (ih iw) c -> b c ih iw', ih=self.ih, iw=self.iw)
        )

    def forward(self, x):
        mbconv = self.MBConv(x)

        if self.downsample:
            xx = self.pool2(x)
            xx = rearrange(xx, 'b c ih iw -> b (ih iw) c')
            norm1 = self.layer_norm(xx)
            attn1 = self.attn(norm1, mbconv)
            attn1 = rearrange(attn1, 'b (ih iw) c -> b c ih iw', ih=self.ih, iw=self.iw)
            out1 = self.proj(self.pool1(x)) + attn1
        else:
            xx = rearrange(x, 'b c ih iw -> b (ih iw) c')
            norm1 = self.layer_norm(xx)
            attn1 = self.attn(norm1, mbconv)
            attn1 = rearrange(attn1, 'b (ih iw) c -> b c ih iw', ih=self.ih, iw=self.iw)
            out1 = x + attn1

        x = out1 + self.ff(out1)
        return x


**10. QUATERNION LAYERS**

In [ ]:
class QuaternionLinear(nn.Module):
    """
    Quaternion-structured linear layer using Hamilton-style interactions.
    Expects input/output dimensions divisible by 4.
    """
    def __init__(self, in_features, out_features):
        super().__init__()
        assert in_features % 4 == 0, 'in_features must be divisible by 4'
        assert out_features % 4 == 0, 'out_features must be divisible by 4'

        self.in_features_q = in_features // 4
        self.out_features_q = out_features // 4

        self.w_r = nn.Parameter(torch.Tensor(self.in_features_q, self.out_features_q))
        self.w_i = nn.Parameter(torch.Tensor(self.in_features_q, self.out_features_q))
        self.w_j = nn.Parameter(torch.Tensor(self.in_features_q, self.out_features_q))
        self.w_k = nn.Parameter(torch.Tensor(self.in_features_q, self.out_features_q))

        self.bias = nn.Parameter(torch.zeros(out_features))

        nn.init.xavier_uniform_(self.w_r)
        nn.init.xavier_uniform_(self.w_i)
        nn.init.xavier_uniform_(self.w_j)
        nn.init.xavier_uniform_(self.w_k)

    def forward(self, x):
        x_r, x_i, x_j, x_k = torch.chunk(x, 4, dim=1)

        out_r = (
            torch.matmul(x_r, self.w_r)
            - torch.matmul(x_i, self.w_i)
            - torch.matmul(x_j, self.w_j)
            - torch.matmul(x_k, self.w_k)
        )

        out_i = (
            torch.matmul(x_r, self.w_i)
            + torch.matmul(x_i, self.w_r)
            + torch.matmul(x_j, self.w_k)
            - torch.matmul(x_k, self.w_j)
        )

        out_j = (
            torch.matmul(x_r, self.w_j)
            - torch.matmul(x_i, self.w_k)
            + torch.matmul(x_j, self.w_r)
            + torch.matmul(x_k, self.w_i)
        )

        out_k = (
            torch.matmul(x_r, self.w_k)
            + torch.matmul(x_i, self.w_j)
            - torch.matmul(x_j, self.w_i)
            + torch.matmul(x_k, self.w_r)
        )

        out = torch.cat([out_r, out_i, out_j, out_k], dim=1)
        out = out + self.bias
        return out


class QuaternionFeatureAdapter(nn.Module):
    """
    Initial quaternion-aware projection for rotation features.
    """
    def __init__(self, feature_dim):
        super().__init__()
        assert feature_dim % 4 == 0, 'feature_dim must be divisible by 4'
        self.norm = nn.LayerNorm(feature_dim)
        self.qproj = QuaternionLinear(feature_dim, feature_dim)
        self.act = nn.GELU()

    def forward(self, x):
        x = self.norm(x)
        x = self.qproj(x)
        x = self.act(x)
        return x


class QuaternionResidualBlock(nn.Module):
    """
    Residual quaternion-aware block for refining rotation features.
    """
    def __init__(self, dim, dropout=0.1):
        super().__init__()
        assert dim % 4 == 0, 'dim must be divisible by 4'
        self.norm = nn.LayerNorm(dim)
        self.qproj = QuaternionLinear(dim, dim)
        self.act = nn.GELU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = self.norm(x)
        out = self.qproj(out)
        out = self.act(out)
        out = self.dropout(out)
        return x + out

**11. Quaternion-aware  COHATNET**

In [ ]:

class CoHAtNet(nn.Module):
    def __init__(self, image_size, in_channels, num_blocks, channels, num_classes=1000,
                 block_types=['C', 'C', 'H', 'H'], dropout_head=0.1):
        super().__init__()
        ih, iw = image_size
        block = {'C': MBConv, 'T': Transformer, 'H': HTransformer}

        self.s0 = self._make_layer(conv_3x3_bn, in_channels, channels[0], num_blocks[0], (ih // 2, iw // 2))
        self.s1 = self._make_layer(block[block_types[0]], channels[0], channels[1], num_blocks[1], (ih // 4, iw // 4))
        self.s2 = self._make_layer(block[block_types[1]], channels[1], channels[2], num_blocks[2], (ih // 8, iw // 8))
        self.s3 = self._make_layer(block[block_types[2]], channels[2], channels[3], num_blocks[3], (ih // 16, iw // 16))
        self.s4 = self._make_layer(block[block_types[3]], channels[3], channels[4], num_blocks[4], (ih // 32, iw // 32))

        self.pool = nn.AvgPool2d(ih // 32, 1)
        feature_dim = channels[-1]

        # Translation branch (unchanged)
        self.translation_head = nn.Sequential(
            nn.Linear(feature_dim, 512),
            nn.GELU(),
            nn.Dropout(dropout_head),
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Dropout(dropout_head),
            nn.Linear(128, 3)
        )

        # Rotation branch (new quaternion-aware design)
        self.rotation_adapter = QuaternionFeatureAdapter(feature_dim)
        self.rotation_res1 = QuaternionResidualBlock(feature_dim, dropout_head)
        self.rotation_res2 = QuaternionResidualBlock(feature_dim, dropout_head)

        self.rotation_head = nn.Sequential(
            QuaternionLinear(feature_dim, 256),
            nn.GELU(),
            nn.Dropout(dropout_head),

            QuaternionLinear(256, 64),
            nn.GELU(),
            nn.Dropout(dropout_head),

            nn.Linear(64, 4)
        )

    def forward(self, x):
        x = self.s0(x)
        x = self.s1(x)
        x = self.s2(x)
        x = self.s3(x)
        x = self.s4(x)

        x = self.pool(x).view(-1, x.shape[1])

        # Translation
        t = self.translation_head(x)

        # Rotation
        q_features = self.rotation_adapter(x)
        q_features = self.rotation_res1(q_features)
        q_features = self.rotation_res2(q_features)
        q = self.rotation_head(q_features)

        # IMPORTANT: normalize quaternion to unit norm
        q = F.normalize(q, p=2, dim=1)

        return torch.cat((t, q), dim=1)

    def _make_layer(self, block, inp, oup, depth, image_size):
        layers = nn.ModuleList([])
        for i in range(depth):
            if i == 0:
                layers.append(block(inp, oup, image_size, downsample=True))
            else:
                layers.append(block(oup, oup, image_size))
        return nn.Sequential(*layers)

def cohatnet_0(dropout_head=0.1):
    num_blocks = [2, 2, 3, 5, 2]
    channels = [64, 96, 192, 384, 768]
    return CoHAtNet((256, 256), 3, num_blocks, channels, num_classes=1000, dropout_head=dropout_head)


**12. SHAPE CHECK**


In [ ]:

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def quick_shape_check():
    img = torch.randn(2, 3, 256, 256)
    net = cohatnet_0()
    out = net(img)
    print('Quick shape check output:', out.shape)
    print('Trainable params:', count_parameters(net))


def get_geo_weight(epoch, start_epoch=15, ramp_epochs=10, max_weight=0.1):
    """
    Curriculum for geometric loss.
    """
    if epoch < start_epoch:
        return 0.0
    elif epoch < start_epoch + ramp_epochs:
        return max_weight * float(epoch - start_epoch + 1) / float(ramp_epochs)
    else:
        return max_weight


def save_checkpoint(path, model, optimizer, epoch, metrics, mean_t, std_t):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'metrics': metrics,
        'mean_t': mean_t,
        'std_t': std_t
    }, path)


def plot_training_curves(history, save_dir, fold_idx):
    epochs = range(1, len(history['train_total_loss']) + 1)

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history['train_total_loss'], label='Train Total Loss')
    plt.plot(epochs, history['val_total_loss'], label='Val Total Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(f'Fold {fold_idx} - Total Loss')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f'fold_{fold_idx}_loss.png'))
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history['train_trans_err'], label='Train Translation Error')
    plt.plot(epochs, history['val_trans_err'], label='Val Translation Error')
    plt.xlabel('Epoch')
    plt.ylabel('Meters')
    plt.title(f'Fold {fold_idx} - Translation Error')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f'fold_{fold_idx}_translation.png'))
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history['train_rot_err'], label='Train Rotation Error')
    plt.plot(epochs, history['val_rot_err'], label='Val Rotation Error')
    plt.xlabel('Epoch')
    plt.ylabel('Degrees')
    plt.title(f'Fold {fold_idx} - Rotation Error')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f'fold_{fold_idx}_rotation.png'))
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history['lr'], label='Learning Rate')
    plt.xlabel('Epoch')
    plt.ylabel('LR')
    plt.title(f'Fold {fold_idx} - Learning Rate')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f'fold_{fold_idx}_lr.png'))
    plt.close()


**13. TRAINING LOOP**

In [ ]:

if __name__ == '__main__':
    root_dir = '/content/drive/MyDrive/7-Scense/fire'
    save_root = '/content/drive/MyDrive/7-Scense/fire_training_outputs'
    os.makedirs(save_root, exist_ok=True)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Using device: {device}')

    base_dataset = BaseFireDataset(root_dir=root_dir)

    groups = np.array(base_dataset.sample_groups)
    unique_groups = np.unique(groups)

    if len(unique_groups) < 2:
        raise ValueError("Need at least 2 sequences/groups for GroupKFold.")

    n_splits = min(5, len(unique_groups))
    group_kfold = GroupKFold(n_splits=n_splits)
    split_iterator = group_kfold.split(np.arange(len(base_dataset)), groups=groups)

    num_epochs = 200

    lambda_trans = 1.0
    lambda_rot = 5.0
    geo_max_weight = 0.03

    geo_start_epoch = 10
    geo_ramp_epochs = 10

    initial_lr = 7e-5
    weight_decay = 1e-4
    warmup_epochs = 5

    batch_size = 8
    grad_accum_steps = 2

    for fold, (train_idx, val_idx) in enumerate(split_iterator):
        fold_num = fold + 1
        fold_dir = os.path.join(save_root, f'fold_{fold_num}')
        os.makedirs(fold_dir, exist_ok=True)

        train_sequences = sorted(set(groups[train_idx]))
        val_sequences = sorted(set(groups[val_idx]))

        print('\n============================================================')
        print(f'Fold {fold_num}/{n_splits}')
        print(f'Train sequences: {train_sequences}')
        print(f'Validation sequences: {val_sequences}')
        print(f'Number of training samples: {len(train_idx)}')
        print(f'Number of validation samples: {len(val_idx)}')
        print('============================================================\n')

        mean_t, std_t = compute_train_translation_stats(base_dataset, train_idx)
        xmin, xmax = compute_train_depth_stats(base_dataset, train_idx)

        train_dataset = FoldFireDataset(
            base_dataset=base_dataset,
            indices=train_idx,
            mean_t=mean_t,
            std_t=std_t,
            transform=transform
        )

        val_dataset = FoldFireDataset(
            base_dataset=base_dataset,
            indices=val_idx,
            mean_t=mean_t,
            std_t=std_t,
            transform=transform
        )

        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            num_workers=0
        )

        val_loader = DataLoader(
            val_dataset,
            batch_size=batch_size,
            shuffle=False,
            num_workers=0
        )

        model = cohatnet_0(dropout_head=0.1).to(device)

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=initial_lr,
            weight_decay=weight_decay
        )

        warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
            optimizer,
            start_factor=0.1,
            end_factor=1.0,
            total_iters=warmup_epochs
        )

        cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=max(1, num_epochs - warmup_epochs),
            eta_min=1e-6
        )

        global_loss_fn = GlobalHomographyLoss(
            xmin=xmin,
            xmax=xmax,
            device=device
        )

        best_val_loss = float('inf')
        best_trans_err = float('inf')
        best_rot_err = float('inf')
        best_epoch = -1

        history = {
            'train_total_loss': [],
            'val_total_loss': [],
            'train_trans_err': [],
            'val_trans_err': [],
            'train_rot_err': [],
            'val_rot_err': [],
            'lr': []
        }

        for epoch in range(num_epochs):
            geo_weight = get_geo_weight(
                epoch=epoch,
                start_epoch=geo_start_epoch,
                ramp_epochs=geo_ramp_epochs,
                max_weight=geo_max_weight
            )

            model.train()
            train_total_loss_sum = 0.0
            train_trans_err_sum = 0.0
            train_rot_err_sum = 0.0
            train_batches = 0

            optimizer.zero_grad()

            for step, (color_images, poses) in enumerate(
                tqdm(train_loader, desc=f'Fold {fold_num} | Epoch {epoch + 1} Train')
            ):
                color_images = color_images.to(device)
                poses = poses.to(device)

                outputs = model(color_images)

                pred_translation_norm = outputs[:, :3]
                pred_quaternion = F.normalize(outputs[:, 3:], p=2, dim=-1, eps=1e-8)
                gt_quaternion = F.normalize(poses[:, 3:], p=2, dim=-1, eps=1e-8)

                pred_translation = denormalize_translation(pred_translation_norm, mean_t, std_t)
                gt_translation = denormalize_translation(poses[:, :3], mean_t, std_t)

                batch = {
                    'w_t_c': pred_translation.unsqueeze(-1),
                    'c_R_w': quaternion_to_rotation_matrix(pred_quaternion),
                    'w_t_chat': gt_translation.unsqueeze(-1),
                    'chat_R_w': quaternion_to_rotation_matrix(gt_quaternion)
                }

                trans_loss = F.smooth_l1_loss(pred_translation, gt_translation)

                rot_loss = 1.0 - torch.abs(torch.sum(pred_quaternion * gt_quaternion, dim=1))
                rot_loss = rot_loss.mean()

                geo_loss = global_loss_fn(batch)

                total_loss = (
                    lambda_trans * trans_loss +
                    lambda_rot * rot_loss +
                    geo_weight * geo_loss
                )

                loss_for_backward = total_loss / grad_accum_steps
                loss_for_backward.backward()

                if ((step + 1) % grad_accum_steps == 0) or ((step + 1) == len(train_loader)):
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()
                    optimizer.zero_grad()

                trans_err = torch.norm(pred_translation - gt_translation, dim=1).mean().item()

                dot_product = torch.abs(torch.sum(pred_quaternion * gt_quaternion, dim=1))
                dot_product = torch.clamp(dot_product, -1.0 + 1e-7, 1.0 - 1e-7)
                rot_err_rad = 2 * torch.acos(dot_product)
                rot_err_deg = torch.rad2deg(rot_err_rad).mean().item()

                train_total_loss_sum += total_loss.item()
                train_trans_err_sum += trans_err
                train_rot_err_sum += rot_err_deg
                train_batches += 1

            avg_train_total_loss = train_total_loss_sum / max(train_batches, 1)
            avg_train_trans_err = train_trans_err_sum / max(train_batches, 1)
            avg_train_rot_err = train_rot_err_sum / max(train_batches, 1)

            model.eval()
            val_total_loss_sum = 0.0
            val_trans_err_sum = 0.0
            val_rot_err_sum = 0.0
            val_batches = 0

            with torch.no_grad():
                for color_images, poses in val_loader:
                    color_images = color_images.to(device)
                    poses = poses.to(device)

                    outputs = model(color_images)

                    pred_translation_norm = outputs[:, :3]
                    pred_quaternion = F.normalize(outputs[:, 3:], p=2, dim=-1, eps=1e-8)
                    gt_quaternion = F.normalize(poses[:, 3:], p=2, dim=-1, eps=1e-8)

                    pred_translation = denormalize_translation(pred_translation_norm, mean_t, std_t)
                    gt_translation = denormalize_translation(poses[:, :3], mean_t, std_t)

                    batch = {
                        'w_t_c': pred_translation.unsqueeze(-1),
                        'c_R_w': quaternion_to_rotation_matrix(pred_quaternion),
                        'w_t_chat': gt_translation.unsqueeze(-1),
                        'chat_R_w': quaternion_to_rotation_matrix(gt_quaternion)
                    }

                    trans_loss = F.smooth_l1_loss(pred_translation, gt_translation)

                    rot_loss = 1.0 - torch.abs(torch.sum(pred_quaternion * gt_quaternion, dim=1))
                    rot_loss = rot_loss.mean()

                    geo_loss = global_loss_fn(batch)

                    total_loss = (
                        lambda_trans * trans_loss +
                        lambda_rot * rot_loss +
                        geo_weight * geo_loss
                    )

                    trans_err = torch.norm(pred_translation - gt_translation, dim=1).mean().item()

                    dot_product = torch.abs(torch.sum(pred_quaternion * gt_quaternion, dim=1))
                    dot_product = torch.clamp(dot_product, -1.0 + 1e-7, 1.0 - 1e-7)
                    rot_err_rad = 2 * torch.acos(dot_product)
                    rot_err_deg = torch.rad2deg(rot_err_rad).mean().item()

                    val_total_loss_sum += total_loss.item()
                    val_trans_err_sum += trans_err
                    val_rot_err_sum += rot_err_deg
                    val_batches += 1

            avg_val_total_loss = val_total_loss_sum / max(val_batches, 1)
            avg_val_trans_err = val_trans_err_sum / max(val_batches, 1)
            avg_val_rot_err = val_rot_err_sum / max(val_batches, 1)

            if epoch < warmup_epochs:
                warmup_scheduler.step()
            else:
                cosine_scheduler.step()

            current_lr = optimizer.param_groups[0]['lr']

            history['train_total_loss'].append(avg_train_total_loss)
            history['val_total_loss'].append(avg_val_total_loss)
            history['train_trans_err'].append(avg_train_trans_err)
            history['val_trans_err'].append(avg_val_trans_err)
            history['train_rot_err'].append(avg_train_rot_err)
            history['val_rot_err'].append(avg_val_rot_err)
            history['lr'].append(current_lr)

            if avg_val_total_loss < best_val_loss:
                best_val_loss = avg_val_total_loss
                best_epoch = epoch + 1
                save_checkpoint(
                    os.path.join(fold_dir, 'best_val_loss.pt'),
                    model, optimizer, epoch + 1,
                    {
                        'val_loss': avg_val_total_loss,
                        'val_trans_err': avg_val_trans_err,
                        'val_rot_err': avg_val_rot_err
                    },
                    mean_t, std_t
                )

            if avg_val_trans_err < best_trans_err:
                best_trans_err = avg_val_trans_err
                save_checkpoint(
                    os.path.join(fold_dir, 'best_translation.pt'),
                    model, optimizer, epoch + 1,
                    {
                        'val_loss': avg_val_total_loss,
                        'val_trans_err': avg_val_trans_err,
                        'val_rot_err': avg_val_rot_err
                    },
                    mean_t, std_t
                )

            if avg_val_rot_err < best_rot_err:
                best_rot_err = avg_val_rot_err
                save_checkpoint(
                    os.path.join(fold_dir, 'best_rotation.pt'),
                    model, optimizer, epoch + 1,
                    {
                        'val_loss': avg_val_total_loss,
                        'val_trans_err': avg_val_trans_err,
                        'val_rot_err': avg_val_rot_err
                    },
                    mean_t, std_t
                )

            print(f'Epoch {epoch + 1}/{num_epochs} | LR: {current_lr:.8f} | Geo Weight: {geo_weight:.4f}')
            print(f'Train Total Loss     : {avg_train_total_loss:.6f}')
            print(f'Train Translation Err: {avg_train_trans_err:.6f} m')
            print(f'Train Rotation Err   : {avg_train_rot_err:.6f} deg')
            print(f'Val Total Loss       : {avg_val_total_loss:.6f}')
            print(f'Val Translation Err  : {avg_val_trans_err:.6f} m')
            print(f'Val Rotation Err     : {avg_val_rot_err:.6f} deg')
            print('-------------------------------------------------------------\n')

            torch.cuda.empty_cache()

        with open(os.path.join(fold_dir, 'history.pkl'), 'wb') as f:
            pickle.dump(history, f)

        with open(os.path.join(fold_dir, 'history.json'), 'w') as f:
            json.dump(history, f, indent=4)

        plot_training_curves(history, fold_dir, fold_num)

        print(f'Fold {fold_num} completed.')
        print(f'Best epoch by val loss : {best_epoch}')
        print(f'Best validation loss   : {best_val_loss:.6f}')
        print(f'Best translation err   : {best_trans_err:.6f} m')
        print(f'Best rotation err      : {best_rot_err:.6f} deg')
        print('===================================================================\n')